In [36]:
import math as m
import numpy as np
import random 
import stable_baselines3 as sb3
#import gym 
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import random 

from abc import ABC, abstractmethod



In [37]:
import copy
import random
import itertools
from abc import ABC, abstractmethod

import numpy as np

In [38]:
import torch
import torch.nn as nn

print(torch.__version__)

net = nn.Linear(4, 4)
#optimizer = torch.optim.SGD(net.parameters(), lr=3e-4)

print("works")

2.13.0+cu130
works


## Building the Env

In [39]:
# redefine box and env  as in box is singular isolated , part of a bigger formula , pattern , structure made by the bigger class .
#test the stuff here 

class CellBox :
    def __init__(
            self,
            name,
            xy:tuple[int,int],
            color:str, 
            symbol:str=None, 
            dirlist : list = None, 
        ):

        # this part describes the box itself
        self.name=name
        self.coordinate=xy
        self.color=color

        self.board = None
        
        
        self.symbol= symbol
        #this one is about the surrounding of the box
        if dirlist == None : dirlist = [None]*4

        self.neighbours={
            "up":dirlist[0],
            "down":dirlist[1],
            "right":dirlist[2],
            "left":dirlist[3],
            "stand":self
        }

    def next_state(self,action):
        if self.neighbours[action] is None:
            return self
        return self.neighbours[action] 
    
    def detach(self):
        self.board = None
    
    # 🔥 KEY FIX: equality based on identity of state meaning
    def __eq__(self, other):
        return isinstance(other, CellBox) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

In [40]:
class Board(ABC):

    def __init__(
        self,
        name: str,
        cells: list[CellBox],
        geometry="box",
        data=None
    ):

        self.id = name
        self.cells = cells
        self.geometry = geometry
        self.data = data

        self.label()
        self.form()

    def label(self):

        for cell in self.cells:

            cell.detach()
            cell.board = self

    def form(self):

        if self.geometry == "box":
            self.make_box(self.data)

        elif self.geometry == "pyramid":
            self.make_pyramid(self.data)

        elif self.geometry == "stack":
            self.make_stack(self.data)

        elif self.geometry == "h_line":
            self.make_line({"direction": "h"})

        elif self.geometry == "v_line":
            self.make_line({"direction": "v"})

        elif self.geometry == "custom":
            self.make_custom(self.data)

        else:
            raise ValueError(
                f"Unknown geometry: {self.geometry}"
            )

    def __iter__(self):

        return iter(self.cells)

    def __len__(self):

        return len(self.cells)

    def snapshot(self):

        return self.cells.copy()

    # --------------------------------------------------
    # BOX
    # --------------------------------------------------

    def make_box(self, data):

        shape = data["shape"]

        N, M = shape

        cells = self.cells

        for m in range(M):

            for n in range(N):

                p = n + m * N

                cell = cells[p]

                cell.neighbours = {
                    "up": None,
                    "down": None,
                    "right": None,
                    "left": None,
                    "stand": cell
                }

                if n < N - 1:
                    cell.neighbours["right"] = cells[p + 1]

                if n > 0:
                    cell.neighbours["left"] = cells[p - 1]

                if m < M - 1:
                    cell.neighbours["up"] = cells[p + N]

                if m > 0:
                    cell.neighbours["down"] = cells[p - N]

    # --------------------------------------------------
    # STACK
    # --------------------------------------------------

    def make_stack(self, data):

        for i in range(len(self.cells)):

            cell = self.cells[i]

            cell.neighbours = {
                "up": None,
                "down": None,
                "right": None,
                "left": None,
                "stand": cell
            }

            if i < len(self.cells) - 1:
                cell.neighbours["up"] = self.cells[i + 1]

            if i > 0:
                cell.neighbours["down"] = self.cells[i - 1]

    # --------------------------------------------------
    # PYRAMID
    # --------------------------------------------------

    def make_pyramid(self, data):

        rows = data["rows"]

        index = 0

        row_cells = []

        for row in range(1, rows + 1):

            current_row = []

            for _ in range(row):

                current_row.append(self.cells[index])

                index += 1

            row_cells.append(current_row)

        # Reset neighbours
        for cell in self.cells:

            cell.neighbours = {
                "up": None,
                "down": None,
                "right": None,
                "left": None,
                "stand": cell
            }

        # Horizontal connections
        for row in row_cells:

            for i in range(len(row) - 1):

                row[i].neighbours["right"] = row[i + 1]
                row[i + 1].neighbours["left"] = row[i]

        # Vertical connections
        for r in range(len(row_cells) - 1):

            current = row_cells[r]
            below = row_cells[r + 1]

            for i, cell in enumerate(current):

                if i < len(below):

                    cell.neighbours["up"] = below[i]
                    below[i].neighbours["down"] = cell

    # --------------------------------------------------
    # LINE
    # --------------------------------------------------

    def make_line(self, data):

        direction = data["direction"]

        if direction == "h":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["right"] = self.cells[i + 1]

                self.cells[i + 1].neighbours["left"] = self.cells[i]

        elif direction == "v":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["up"] = self.cells[i + 1]

                self.cells[i + 1].neighbours["down"] = self.cells[i]

        else:

            raise ValueError(
                "direction must be 'h' or 'v'"
            )

    # --------------------------------------------------
    # CUSTOM
    # --------------------------------------------------

    def make_custom(self, data):

        opposite = {
            "right": "left",
            "left": "right",
            "up": "down",
            "down": "up"
        }

        for cell_a, direction, cell_b in data["connections"]:

            if direction not in opposite:

                raise ValueError(
                    f"Invalid direction: {direction}"
                )

            cell_a.neighbours[direction] = cell_b

            cell_b.neighbours[opposite[direction]] = cell_a

In [41]:
# Env (organizing the puzzle of state into env )
class Env:

    @property
    def actions(self) -> list:
        """All actions available in this environment."""
        raise NotImplementedError

    @property
    def states(self) -> list:
        """All states in this environment."""
        raise NotImplementedError

    def reset(self):
        """Start a new episode. Return the initial state."""
        raise NotImplementedError

    def step(self, action):
        """
        Apply action to the current state.
        Return (next_state, reward, done).
        """
        raise NotImplementedError
    

In [42]:
class Chessboard(Env):
    """
    RL environment.
    Board        → geometry / topology
    Chessboard   → transitions + reward + terminal condition
    """

    def __init__(
        self,
        board: Board,
        actions: list[str],
        start: CellBox,
        reward_fn=None
    ):
        self.board = board
        self.id = board.id
        self._actions = actions
        self.start = start
        self.current = start

        # Make sure start is not a terminal cell
        if self.start.symbol in ("X", "O"):
            self.start.symbol = None

        # Reward function (injectable)
        if reward_fn is None:
            self.reward_fn = self.default_reward
        else:
            self.reward_fn = reward_fn

    @property
    def states(self):
        return self.board.cells

    @property
    def actions(self):
        return self._actions

    def default_reward(self, state, action, next_state):
        if next_state.symbol == "X":
            return -100
        elif next_state.symbol == "O":
            return 100
        else:
            return -10

    def reset(self):
        self.current = self.start
        return self.current

    def step(self, action):
        state = self.current
        next_state = state.next_state(action)

        # Reward is now calculated by the injected function
        reward = float(self.reward_fn(state, action, next_state))

        done = next_state.symbol in ("X", "O")
        self.current = next_state

        return {
            "state": state,
            "action": action,
            "reward": reward,
            "next_state": next_state,
            "done": done,
        }

    def snapshot(self):
        return {
            "current": self.current,
            "start": self.start,
            "board": self.board.snapshot(),
        }

In [43]:
# some various reward function

# Version 1 – classic
def reward_v1(state, action, next_state):
    if next_state.symbol == "O":
        return 100
    if next_state.symbol == "X":
        return -100
    return -10




# Version 2 – stronger penalties
def reward_v2(state, action, next_state):
    if next_state.symbol == "O":
        return 500
    if next_state.symbol == "X":
        return -500
    return -1




# Version 3 – distance-based shaping
def reward_v3(state, action, next_state):

    # Terminal rewards
    if next_state.symbol == "O":
        return 10.0

    if next_state.symbol == "X":
        return -10.0


    # Get all goals
    goals = [
        cell for cell in state.board.cells
        if cell.symbol == "O"
    ]

    # Safety fallback
    if not goals:
        return -0.1


    # Distance BEFORE the action
    old_dist = min(
        abs(state.coordinate[0] - g.coordinate[0]) +
        abs(state.coordinate[1] - g.coordinate[1])
        for g in goals
    )


    # Distance AFTER the action
    new_dist = min(
        abs(next_state.coordinate[0] - g.coordinate[0]) +
        abs(next_state.coordinate[1] - g.coordinate[1])
        for g in goals
    )


    # Movement reward
    if new_dist < old_dist:
        return 0.5       # moved closer

    elif new_dist > old_dist:
        return -0.5      # moved farther

    else:
        return -0.1      # no progress

def reward_v4(state, action ,next_state):

    # =========================================================
    # 1. TERMINAL REWARDS
    # =========================================================
    if next_state.symbol == "O":
        return 1.0

    if next_state.symbol == "X":
        return -1.0

    # =========================================================
    # 2. DISTANCE PROGRESS
    # =========================================================
    goals = [
        cell for cell in state.board.cells
        if cell.symbol == "O"
    ]

    if not goals:
        return -0.01

    old_dist = min(
        manhattan(state, goal)
        for goal in goals
    )

    new_dist = min(
        manhattan(next_state, goal)
        for goal in goals
    )

    progress = old_dist - new_dist

    # =========================================================
    # 3. DENSE REWARD
    # =========================================================
    progress_bonus = 0.1 * progress
    step_penalty = -0.01

    return progress_bonus + step_penalty

### Setting an example board

In [44]:
#set up
s1 = CellBox("s1",(0,0),"white","X",[None,None,None,None])
s2 = CellBox("s2",(2,0),"white","O",[None,None,None,None])
s3 = CellBox("s3",(4,0),"white","X",[None,None,None,None])
s4 = CellBox("s4",(0,1),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"grey",None,[None,None,None,None])
s6 = CellBox("s6",(2,1),"white",None,[None,None,None,None])
s7 = CellBox("s7",(3,1),"grey",None,[None,None,None,None])
s8 = CellBox("s8",(4,1),"white",None,[None,None,None,None])

#s0 = CellBox("empty box")

#define
s1.neighbours = {"up": s4,   "down": None, "right": None, "left": None, "stand": s1}
s2.neighbours = {"up": s6,   "down": None, "right": None, "left": None, "stand": s2}
s3.neighbours = {"up": s8,   "down": None, "right": None, "left": None, "stand": s3}
s4.neighbours = {"up": None, "down": s1,   "right": s5,   "left": None, "stand": s4}
s5.neighbours = {"up": None, "down": None, "right": s6,   "left": s4,   "stand": s5}
s6.neighbours = {"up": None, "down": s2,   "right": s7,   "left": s5,   "stand": s6}
s7.neighbours = {"up": None, "down": None, "right": s8,   "left": s6,   "stand": s7}
s8.neighbours = {"up": None, "down": s3,   "right": None, "left": s7,   "stand": s8}
#fwefsdf
States1=[s1,s2,s3,s4,s5,s6,s7,s8]

Actions=["up", "right","down","left"]




In [45]:
s1 = CellBox("s1",(2,0),"white",None,[None,None,None,None])
s2 = CellBox("s2",(2,1),"white",None,[None,None,None,None])
s3 = CellBox("s3",(2,2),"white",None,[None,None,None,None])
s4 = CellBox("s4",(1,0),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"white","X",[None,None,None,None])
s6 = CellBox("s6",(1,2),"white",None,[None,None,None,None])
s7 = CellBox("s7",(0,0),"white",None,[None,None,None,None])
s8 = CellBox("s8",(0,1),"white",None,[None,None,None,None])
s9 = CellBox("s9",(0,2),"white","O",[None,None,None,None])

States2 = [s1,s2,s3,s4,s5,s6,s7,s8,s9]

claude_board = Board("claude_board",States2,geometry="box",data={"shape":(3,3)})

env = Chessboard(
    board=claude_board,
    actions=Actions,
    start=s1
)

state = env.reset()

experience = env.step("right")

In [46]:
experience["next_state"].name

isinstance(env, Env)

True

## bigger env

In [47]:
b11 = CellBox("b11",(0,0),"white",None,[None,None,None,None])
b12 = CellBox("b12",(0,1),"white",None,[None,None,None,None])
b13 = CellBox("b13",(0,2),"white",None,[None,None,None,None])
b14 = CellBox("b14",(1,0),"white",None,[None,None,None,None])
b15 = CellBox("b15",(1,1),"white",None,[None,None,None,None])
b16 = CellBox("b16",(1,2),"white",None,[None,None,None,None])
b17 = CellBox("b17",(2,0),"white",None,[None,None,None,None])
b18 = CellBox("b18",(2,1),"white",None,[None,None,None,None])
b19 = CellBox("b19",(2,2),"white",None,[None,None,None,None])

Block1 = [b11, b12 , b13 , b14 ,b15 , b16 , b17 , b18 , b19 ]
Board1 = Board("Board1",Block1 , geometry= "box", data= {"shape":(3,3)})

b21 = CellBox("b21",(3,0),"white",None,[None,None,None,None])
b22 = CellBox("b22",(3,1),"white",None,[None,None,None,None])
b23 = CellBox("b23",(3,2),"white",None,[None,None,None,None])
b24 = CellBox("b24",(4,0),"white",None,[None,None,None,None])
b25 = CellBox("b25",(4,1),"white",None,[None,None,None,None])
b26 = CellBox("b26",(4,2),"white",None,[None,None,None,None])
b27 = CellBox("b27",(5,0),"white",None,[None,None,None,None])
b28 = CellBox("b28",(5,1),"white",None,[None,None,None,None])
b29 = CellBox("b29",(5,2),"white",None,[None,None,None,None])

Block2 = [b21, b22 , b23 , b24 ,b25 , b26 , b27 , b28 , b29 ]
Board2 = Board("Board2",Block2 , geometry= "box", data= {"shape":(3,3)})

b31 = CellBox("b31",(0,3),"white",None,[None,None,None,None])
b32 = CellBox("b32",(0,4),"white",None,[None,None,None,None])
b33 = CellBox("b33",(0,5),"white",None,[None,None,None,None])
b34 = CellBox("b34",(1,3),"white",None,[None,None,None,None])
b35 = CellBox("b35",(1,4),"white",None,[None,None,None,None])
b36 = CellBox("b36",(1,5),"white",None,[None,None,None,None])
b37 = CellBox("b37",(2,3),"white",None,[None,None,None,None])
b38 = CellBox("b38",(2,4),"white",None,[None,None,None,None])
b39 = CellBox("b39",(2,5),"white",None,[None,None,None,None])

Block3 = [b31, b32 , b33 , b34 ,b35 , b36 , b37 , b38 , b39 ]
Board3 = Board("Board3",Block3 , geometry= "box", data= {"shape":(3,3)})

b41 = CellBox("b41",(3,3),"white",None,[None,None,None,None])
b42 = CellBox("b42",(3,4),"white",None,[None,None,None,None])
b43 = CellBox("b43",(3,5),"white",None,[None,None,None,None])
b44 = CellBox("b44",(4,3),"white",None,[None,None,None,None])
b45 = CellBox("b45",(4,4),"white",None,[None,None,None,None])
b46 = CellBox("b46",(4,5),"white",None,[None,None,None,None])
b47 = CellBox("b47",(5,3),"white",None,[None,None,None,None])
b48 = CellBox("b48",(5,4),"white",None,[None,None,None,None])
b49 = CellBox("b49",(5,5),"white",None,[None,None,None,None])

Block4 = [b41, b42 , b43 , b44 ,b45 , b46 , b47 , b48 , b49 ]
Board4 = Board("Board4",Block4 , geometry= "box", data= {"shape":(3,3)})

b51 = CellBox("b51",(0,6),"white",None,[None,None,None,None])
b52 = CellBox("b52",(0,7),"white",None,[None,None,None,None])
b53 = CellBox("b53",(0,8),"white",None,[None,None,None,None])
b54 = CellBox("b54",(1,6),"white",None,[None,None,None,None])
b55 = CellBox("b55",(1,7),"white",None,[None,None,None,None])
b56 = CellBox("b56",(1,8),"white",None,[None,None,None,None])
b57 = CellBox("b57",(2,6),"white",None,[None,None,None,None])
b58 = CellBox("b58",(2,7),"white",None,[None,None,None,None])
b59 = CellBox("b59",(2,8),"white",None,[None,None,None,None])

Block5 = [b51, b52 , b53 , b54 ,b55 , b56 , b57 , b58 , b59 ]
Board5 = Board("Board5",Block5 , geometry= "box", data= {"shape":(3,3)})

b61 = CellBox("b61",(3,6),"white",None,[None,None,None,None])
b62 = CellBox("b62",(3,7),"white",None,[None,None,None,None])
b63 = CellBox("b63",(3,8),"white",None,[None,None,None,None])
b64 = CellBox("b64",(4,6),"white",None,[None,None,None,None])
b65 = CellBox("b65",(4,7),"white",None,[None,None,None,None])
b66 = CellBox("b66",(4,8),"white",None,[None,None,None,None])
b67 = CellBox("b67",(5,6),"white",None,[None,None,None,None])
b68 = CellBox("b68",(5,7),"white",None,[None,None,None,None])
b69 = CellBox("b69",(5,8),"white",None,[None,None,None,None])

Block6 = [b61, b62 , b63 , b64 ,b65 , b66 , b67 , b68 , b69 ]
Board6 = Board("Board6",Block6 , geometry= "box", data= {"shape":(3,3)})


# glew the boards together
Global_board = Board("Global_board",Block1 + Block2 +Block3 + Block4 + Block5 +Block6 , geometry= "custom" ,
                      data= { "connections" :
                             [
                             (b17,"up",b21),(b18,"up",b22),(b19,"up",b23),
                             (b13,"right", b31), (b16,"right", b34), (b19,"right", b37),
                             (b23,"right", b41), (b26,"right", b44), (b29,"right", b47),
                             (b37 ,"up",b41), (b38 ,"up",b42), (b39,"up", b43 ),
                             (b43,"right", b61 ), (b46,"right", b64 ), (b49,"right", b67 ),
                             (b33,"right", b51 ), (b36,"right", b54 ), (b39,"right", b57 ),
                             (b61,"down", b57) , (b62,"down", b58) , (b63,"down", b59),
                             ]
                      }
                    )

# choose randomly which cells are traps and 3 goal points:


Blocks = Block1 + Block2 + Block3 + Block4 + Block5 + Block6
goals = 3
traps = 10
goal_cells, trap_cells, normal_cells = [],[],[]

random.shuffle(Blocks)

goal_cells = Blocks[:goals]
trap_cells = Blocks[goals : goals + traps]
normal_cells  = Blocks[goals + traps :]

# Assign symbols
for cell in goal_cells:
    cell.symbol = "O"

for cell in trap_cells:
    cell.symbol = "X"



# Optional: print summary
print(f"Goals  ({len(goal_cells)}): {[c.name for c in goal_cells]}")
print(f"Traps  ({len(trap_cells)}): {[c.name for c in trap_cells]}")
print(f"Normal ({len(normal_cells)}): {len(normal_cells)} cells")
       


Goals  (3): ['b55', 'b69', 'b21']
Traps  (10): ['b49', 'b24', 'b15', 'b44', 'b68', 'b45', 'b37', 'b25', 'b63', 'b36']
Normal (41): 41 cells


## Define the State vector

In [48]:
def manhattan(a: CellBox, b: CellBox) -> float:
    return float(
        abs(a.coordinate[0] - b.coordinate[0]) +
        abs(a.coordinate[1] - b.coordinate[1])
    )


def one_hot_cell(cell):
    """
    Returns:

    Empty -> [1, 0, 0, 0]
    Wall  -> [0, 1, 0, 0]
    Trap  -> [0, 0, 1, 0]
    Goal  -> [0, 0, 0, 1]
    """

    if cell is None:
        return [0.0, 1.0, 0.0, 0.0]

    if cell.symbol == "X":
        return [0.0, 0.0, 1.0, 0.0]

    if cell.symbol == "O":
        return [0.0, 0.0, 0.0, 1.0]

    return [1.0, 0.0, 0.0, 0.0]


def state_to_vector(state: CellBox) -> np.ndarray:
    """
    State representation:

    LOCAL:
        Immediate neighbour in each direction
        -> empty / wall / trap / goal

    NEAR-LOCAL:
        Second cell in each direction
        -> empty / wall / trap / goal

    GOAL / PROGRESS:
        Progress obtainable by moving in each direction
        -> positive = closer
        -> zero     = no change
        -> negative = farther

        Normalized distance to nearest goal

    Total:
        16 local
        + 16 near-local
        + 4 progress
        + 1 goal distance
        = 37 features
    """

    directions = ["up", "right", "down", "left"]

    # =========================================================
    # 1. GET GOALS
    # =========================================================

    goals = [
        cell for cell in state.board.cells
        if cell.symbol == "O"
    ]

    # =========================================================
    # 2. NORMALIZATION
    # =========================================================

    coords = [
        cell.coordinate
        for cell in state.board.cells
    ]

    if coords:
        max_dist = float(
            max(
                max(abs(x) for x, y in coords),
                max(abs(y) for x, y in coords)
            )
        ) or 1.0
    else:
        max_dist = 1.0

    # =========================================================
    # 3. NEAREST GOAL
    # =========================================================

    if goals:
        nearest_goal = min(
            goals,
            key=lambda g: manhattan(state, g)
        )

        goal_distance = manhattan(
            state,
            nearest_goal
        )

        normalized_goal_distance = (
            goal_distance / max_dist
        )

    else:
        nearest_goal = None
        normalized_goal_distance = 0.0

    # =========================================================
    # 4. LOCAL + NEAR-LOCAL + PROGRESS
    # =========================================================

    local_features = []
    near_local_features = []
    progress_features = []

    current_distance = (
        manhattan(state, nearest_goal)
        if nearest_goal is not None
        else 0.0
    )

    for direction in directions:

        # -----------------------------------------------------
        # Immediate neighbour
        # -----------------------------------------------------

        neighbour = state.neighbours[direction]

        local_features.extend(
            one_hot_cell(neighbour)
        )

        # -----------------------------------------------------
        # Second cell in same direction
        # -----------------------------------------------------

        if neighbour is not None:
            second_neighbour = neighbour.neighbours[direction]
        else:
            second_neighbour = None

        near_local_features.extend(
            one_hot_cell(second_neighbour)
        )

        # -----------------------------------------------------
        # Progress if moving in this direction
        # -----------------------------------------------------

        if (
            nearest_goal is None
            or neighbour is None
            or neighbour.symbol == "X"
        ):
            progress = -1.0

        else:
            neighbour_distance = manhattan(
                neighbour,
                nearest_goal
            )

            progress = (
                current_distance -
                neighbour_distance
            ) / max_dist

        progress_features.append(progress)

    # =========================================================
    # 5. FINAL VECTOR
    # =========================================================

    return np.array(
        local_features
        + near_local_features # see if i can manage and do without it 
        + progress_features
        + [normalized_goal_distance],
        dtype=np.float32
    )

## Policy

In [49]:
from abc import ABC, abstractmethod

class Policy(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def action_distribution(self, state): pass
    @abstractmethod
    def snapshot(self): pass

### Some examples

In [50]:
class DeterministicPolicy(Policy):
    def __init__(self):
        self.mapping = {}
    def select_action(self, state):        return self.mapping[state]
    def action_distribution(self, state):  return {self.mapping[state]: 1.0}
    def snapshot(self):                    return self.mapping.copy()

class StochasticPolicy(Policy):
    def __init__(self):
        self.pi = {}
    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]
    def action_distribution(self, state):  return self.pi[state]
    def snapshot(self):                    return self.pi.copy()

class Policy_theta_softmax(Policy):
    def __init__(self, phi, theta): # phi:FeatureFunction
        self.pi      = {}
        self.weights = theta
        self.phi     = phi

    def _zeller(self, state, action, Actions): #
        n  = Actions.index(action)
        a1 = Actions[(n + 1) % 4]
        a2 = Actions[(n + 3) % 4]
        phi_sa = 0.8 * self.phi.compute(state.next_state(action)) + \
                 0.1 * (self.phi.compute(state.next_state(a1)) +
                        self.phi.compute(state.next_state(a2)))
        h = (self.weights @ phi_sa).item()
        return np.exp(h)

    def calculate_all_probabilities(self, state, Actions):
        # compute each zeller once, reuse for denominator
        zellers = {a: self._zeller(state, a, Actions) for a in Actions}
        N = sum(zellers.values())
        self.pi[state] = {a: float(z / N) for a, z in zellers.items()}

    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]

    def action_distribution(self, state):  return self.pi.get(state, {})
    def snapshot(self):                    return self.pi.copy()

class NeuralPolicy(Policy):
    def __init__(self, network):
        self.network = network
    def select_action(self, state):
        probs = self.network(state)
        return 0  # placeholder
    def action_distribution(self, state):  return self.network(state)
    def snapshot(self):                    return self.network.state_dict()

## Learning Strategy (for improvement and update)

In [51]:
from abc import ABC, abstractmethod

class LearningStrategy(ABC):
    @abstractmethod
    def update(self, policy, experience): pass
    @abstractmethod
    def snapshot(self): pass

class QLearning(LearningStrategy):
    def __init__(self, actions, alpha=0.1, gamma=0.9):
        self.alpha   = alpha
        self.gamma   = gamma
        self.actions = actions
        self.Q       = {}

    def _ensure(self, s):
        if s not in self.Q:
            self.Q[s] = {a: 0.0 for a in self.actions}

    def update(self, policy, exp):
        s, a, r, next_s, done = exp["state"], exp["action"], exp["reward"], exp["next_state"], exp["done"]
        self._ensure(s)
        self._ensure(next_s)
        target = r if done else r + self.gamma * max(self.Q[next_s].values())
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])
        policy.mapping[s] = max(self.Q[s], key=self.Q[s].get)

    def snapshot(self):
        return self.Q.copy()

class Reinforce(LearningStrategy):
    def __init__(self, lr=0.1):
        self.lr = lr
    def update(self, policy, state_vec, action_index, reward):
        probs = policy.forward(state_vec)
        for i in range(policy.W.shape[1]):
            policy.W[0, i] += self.lr * reward * ((1 if i == action_index else 0) - probs[i]) * state_vec[0]
    def snapshot(self): return {}



## Generalized Advantage Estimation

In [52]:
def compute_gae(deltas, gamma=0.99, lam=0.95):
    """Compute GAE advantages from a list of TD residuals (deltas)"""
    advantages = torch.zeros(len(deltas))
    gae = torch.tensor(0.0)
    
    # Backward pass - this is the key
    for t in reversed(range(len(deltas))):
        gae = deltas[t] + gamma * lam * gae
        advantages[t] = gae
    
    return advantages


### example


In [53]:
# Simulate adding steps one by one (as in your question)
deltas = []

print("Incremental GAE Update:\n")

for step in range(1, 7):
    # Simulate receiving a new step
    new_delta = round(np.random.uniform(-1.0, 2.0), 3)   # random for demo
    deltas.append(new_delta)
    
    print(f"After step {step} (new δ = {new_delta}):")
    
    # Recompute GAE using ALL data available so far
    advantages = compute_gae(deltas, gamma=0.99, lam=0.95)
    
    for i, adv in enumerate(advantages):
        print(f"   A[{i+1:2d}] = {adv:.3f}")
    print("-" * 50)

Incremental GAE Update:

After step 1 (new δ = 0.549):
   A[ 1] = 0.549
--------------------------------------------------
After step 2 (new δ = 0.897):
   A[ 1] = 1.393
   A[ 2] = 0.897
--------------------------------------------------
After step 3 (new δ = 0.01):
   A[ 1] = 1.401
   A[ 2] = 0.906
   A[ 3] = 0.010
--------------------------------------------------
After step 4 (new δ = 1.187):
   A[ 1] = 2.389
   A[ 2] = 1.956
   A[ 3] = 1.126
   A[ 4] = 1.187
--------------------------------------------------
After step 5 (new δ = -0.926):
   A[ 1] = 1.664
   A[ 2] = 1.186
   A[ 3] = 0.307
   A[ 4] = 0.316
   A[ 5] = -0.926
--------------------------------------------------
After step 6 (new δ = 1.024):
   A[ 1] = 2.418
   A[ 2] = 1.987
   A[ 3] = 1.159
   A[ 4] = 1.222
   A[ 5] = 0.037
   A[ 6] = 1.024
--------------------------------------------------


## Blocks to build Neural Network

In [54]:
# Uncomment if you want to use this simple version Brick 1 and 2 

# ── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
class Block(nn.Module):
    """A reusable residual-style chunk: linear → norm → activation"""
    def __init__(self, dim, activation=nn.ReLU):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act  = activation()

    def forward(self, x):
        return self.act(self.norm(self.fc(x)))

# ── BRICK 2: shared body ─────────────────────────────────────────────────
class SharedBody(nn.Module):
    """
    Encodes the state φ(s) into a shared representation.
    Both actor and critic read from this.
    """
    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 64,
        n_layers: int = 2,
        activation=nn.ReLU
    ):
        super().__init__()

        self.input_layer = nn.Linear(n_features, hidden_dim)

        layers = [
            self.input_layer,
            activation()
        ]

        for i in range(1, n_layers + 1):
            setattr(self, f"block{i}", Block(hidden_dim, activation))

        for i in range(1, n_layers + 1):
            layers.append(getattr(self, f"block{i}"))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# class SharedBody(nn.Module): # efficient version
#     def __init__(
#         self,
#         n_layers: int,
#         n_features: int,
#         hidden_dim: int = 64,
#         activation=nn.ReLU
#     ):
#         super().__init__()

#         self.network = nn.Sequential(
#             nn.Linear(n_features, hidden_dim),
#             activation(),
#             *[Block(hidden_dim, activation) for _ in range(n_layers)]
#         )

        
#vars() was useful here 

In [55]:
# # Uncomment if you want to use this simple version Brick 1 and 2 

#── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
# class Block(nn.Module):
#     """A reusable residual-style chunk: linear → norm → relu"""
#     def __init__(self, dim):
#         super().__init__()
#         self.fc   = nn.Linear(dim, dim)
#         self.norm = nn.LayerNorm(dim)

#     def forward(self, x):
#         return F.relu(self.norm(self.fc(x)))  # ← fixed: F.relu() not nn.ReLU()


#── BRICK 2: shared body ─────────────────────────────────────────────────
# class SharedBody(nn.Module):
#     """
#     Encodes the state φ(s) into a shared representation.
#     Both actor and critic read from this — they see the same features.
#     Input:  φ(s) of shape (n_features,)
#     Output: hidden representation of shape (hidden_dim,)
#     """
#     def __init__(self, n_features: int, hidden_dim: int = 64):
#         super().__init__()
#         self.input_layer = nn.Linear(n_features, hidden_dim)
#         self.block1      = Block(hidden_dim)
#         self.block2      = Block(hidden_dim)

#         #self.network = nn.Sequential(....)

#     def forward(self, x):
#         x = F.relu(self.input_layer(x))
#         x = self.block1(x)
#         x = self.block2(x)
#         return x

In [56]:

# ── BRICK 3: actor head ──────────────────────────────────────────────────
class ActorHead(nn.Module):
    """
    Takes shared body output → outputs a probability distribution over actions.
    π_θ(a|s) = softmax(W · h + b)
    """
    def __init__(self, hidden_dim: int, action_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, action_dim)

    def forward(self, h):
        return F.softmax(self.head(h), dim=-1)   # shape: (action_dim,)


class GaussianActor(nn.Module):
    """
    Takes shared body output → outputs a Gaussian probability distribution over continuous actions.
    μ = W · h + b
    σ = exp(log_std)

    π_θ(a|s) = Normal(μ, σ)
    """
    def __init__(self, hidden_dim: int, action_dim: int):
            super().__init__()

            self.mean = nn.Linear(hidden_dim,action_dim)
            self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, h):
        mean = self.mean(h)
        std = torch.exp(self.log_std)

        return torch.distributions.Normal(mean,std)


# ── BRICK 4: critic head ─────────────────────────────────────────────────
class CriticHead(nn.Module):
    """
    Takes shared body output → outputs a single scalar V(s).
    No activation — value can be any real number.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        return self.head(h).squeeze(-1)           # shape: scalar


# ── BRICK 5: full Actor-Critic network ───────────────────────────────────
class ActorCritic(nn.Module):
    """
    One network, two outputs:
        actor  → π_θ(a|s)   used for: action selection + policy loss
        critic → V_θ(s)     used for: GAE computation  + value loss
    """
    def __init__(self, 
                n_features: int,
                action_dim: int, 
                hidden_dim: int = 64,
                n_layers = 2,
                action_mode = "discrete"):
        super().__init__()
        self.action_mode = action_mode
        self.body   = SharedBody(n_features, hidden_dim,n_layers)
        self.critic = CriticHead(hidden_dim)
        if action_mode == "discrete":
            self.actor  = ActorHead(hidden_dim, action_dim)
        elif (action_mode== "continuous") or (action_mode == "Gauss"):
            self.actor  = GaussianActor(hidden_dim, action_dim)

        else: 
            raise ValueError(
                "actor_mode must be 'discrete' or 'continous'|'Gauss'"
            )

        self._init_weights()

    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        # small gain → initial policy near-uniform → more exploration
        nn.init.orthogonal_(self.critic.head.weight, gain=1.0)
        nn.init.constant_(self.critic.head.bias, 0.0)

        if self.action_mode == "discrete":

            nn.init.orthogonal_(self.actor.head.weight,gain=0.01)
            nn.init.constant_(self.actor.head.bias,0.0)
        else: # the case where gauß is needed
            nn.init.orthogonal_(self.actor.mean.weight,gain=0.01)
            nn.init.constant_(self.actor.mean.bias,0.0)


    def forward(self, x):
        h     = self.body(x)
        value = self.critic(h)   # V_θ(s)
        #probs = self.actor(h)    # π_θ(a|s)

        if self.action_mode == "discrete":

            probs = self.actor(h)    # π_θ(a|s)
            dist = torch.distributions.Categorical(probs)
        else : 
            pdist = self.actor(h)    # π_θ(a|s)
        
        return dist, value

    def get_action(self, state_vec: np.ndarray):

        x = torch.tensor(state_vec,dtype=torch.float32)
        dist, value = self.forward(x)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        if self.action_mode == "continuous":
            log_prob = log_prob.sum()

        if self.action_mode == "discrete":
            action = action.item()

        else:
            action = action.detach().numpy()

        return action, log_prob, value

# ── BRICK 6: your custom loss (kept from your code) ──────────────────────
class MyLoss(nn.Module):
    """MSE loss — used for critic: (V(s) - y_t)²"""
    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()
    

In [57]:
class Policy_PPO(Policy):
    def __init__(self,network,actions,state_to_vector_fn = state_to_vector):
        super().__init__()
        self.pi ={}
        self.net = network
        self.actions = actions
        self.state_to_vector_fn = state_to_vector_fn
         
    def select_action(self, state):
        x = self.state_to_vector_fn(state)
        action , log_prob , value = self.net.get_action(x) # see what we can do with the value

        if self.net.action_mode == "discrete":
            action = self.actions[action] # here action is action_index

        return action , log_prob, value
    def action_distribution(self, state):
        x = torch.tensor(self.state_to_vector_fn(state),dtype=torch.float32)

        with torch.no_grad():
            dist, _ = self.net(x)
            
        return dist
    

    def snapshot(self): return self.net.state_dict()
        

### Examples for sanity check

In [58]:
# ── quick sanity check ───────────────────────────────────────────────────
n_features = 2   # [dist_goal, dist_trap] — your phi(s) size
action_dim  = 4   # up, right, down, left

net = ActorCritic(n_features=n_features, action_dim=action_dim, hidden_dim=64, n_layers= 100)

dummy_state = torch.tensor([2.0, 1.0], dtype=torch.float32)
dist, value = net(dummy_state)

print(f"Action probs : {dist.probs.detach().numpy()}")  # 4 numbers summing to 1
print(f"Value        : {value.item():.4f}")        # single scalar
print(f"Total params : {sum(p.numel() for p in net.parameters())}")

# act from a real numpy feature vector
state_vec = np.array([2.0, 1.0])
action_idx, log_prob, val = net.get_action(state_vec) 
print(f"Sampled action index: {action_idx} → {Actions[action_idx]}")




Action probs : [0.2518814  0.24723192 0.25048542 0.25040132]
Value        : 0.2013
Total params : 429317
Sampled action index: 2 → down


In [59]:
n_features = 2
action_dim = 4

net = ActorCritic(
    n_features=n_features,
    action_dim=action_dim,
    hidden_dim=64,
    n_layers=2,
    action_mode="discrete"
)

dummy_state = torch.tensor([2.0, 1.0],dtype=torch.float32)
dist, value = net(dummy_state)

print(f"Action probs : {dist.probs.detach().numpy()}")
print(f"Value        : {value.item():.4f}")
print(f"Total params : {sum(p.numel() for p in net.parameters())}")

state_vec = np.array([2.0, 1.0])

action, log_prob, val = net.get_action(state_vec)

print(f"Sampled action index: {action} → {Actions[action]}")
print(f"Log probability: {log_prob.item():.4f}")
print(f"Value: {val.item():.4f}")

Action probs : [0.25049773 0.2504986  0.2510287  0.24797505]
Value        : 1.1647
Total params : 9093
Sampled action index: 1 → right
Log probability: -1.3843
Value: 1.1647


In [60]:
#vars(net.body)

### example to see if everything works together

In [61]:
# example_state = env.states[0]

# n_features = len(state_to_vector(example_state))
# pi = Policy_PPO(ActorCritic(n_features= n_features , action_dim=action_dim, hidden_dim=64),Actions) 
# pi.select_action(s6) #Actions[a_idx] , log_prob, value

## Defining Loss class for Improvement / Learning / Updating

In [62]:
class Loss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, *args, **kwargs):
        raise NotImplementedError



## My Loss function for PPO

In [79]:
class Loss_Entropy(Loss):
    """Entropy encourages exploration."""
    def forward(self, dist) -> torch.Tensor:
        return dist.entropy().mean()


class Loss_Critic(Loss):
    """L_V = (V(s) - y)²"""
    def forward(
        self,
        v_pred: torch.Tensor,
        v_target: torch.Tensor
    ) -> torch.Tensor:

        return F.mse_loss(v_pred, v_target)


class Loss_Actor(Loss):
    """L_pi = -ρ·A — PPO clipped policy gradient"""

    def __init__(self, clip_epsilon=0.2):
        super().__init__()
        self.eps = clip_epsilon

    def forward(
        self,
        log_prob_new: torch.Tensor,
        log_prob_old: torch.Tensor,
        advantage: torch.Tensor
    ) -> torch.Tensor:

        # ρ = π_new / π_old
        rho = torch.exp(log_prob_new - log_prob_old)

        clipped = torch.clamp(
            rho,
            1 - self.eps,
            1 + self.eps
        )

        # PPO clipped objective
        L_clip = torch.min(
            rho * advantage,
            clipped * advantage
        )

        return -L_clip.mean()


class Loss_PPO(Loss):

    def __init__(self, cv=0.5, ce=0.01, clip_epsilon=0.2):
        super().__init__()

        self.cv = cv
        self.ce = ce

        self.L_pi = Loss_Actor(clip_epsilon)
        self.L_v = Loss_Critic()

    def forward(
        self,
        log_prob_new,
        log_prob_old,
        advantage,
        v_pred,
        v_target,
        entropy
    ):
        actor_loss = self.L_pi(
            log_prob_new,
            log_prob_old,
            advantage
        )

        critic_loss = self.L_v(
            v_pred,
            v_target
        )

        return (
            actor_loss
            + self.cv * critic_loss
            - self.ce * entropy
        )

In [64]:

class PPO_old(LearningStrategy):
    def __init__(self,
                optimizer,
                clip_epsilon: float = 0.2,
                cv:           float = 0.5,    # critic loss weight
                ce:           float = 0.01,   # entropy bonus weight
                gamma:        float = 0.99,
                lam:          float = 0.95,
                n_epochs:     int   = 1):
        self.optimizer    = optimizer
        self.clip_epsilon = clip_epsilon
        self.cv           = cv       
        self.ce           = ce          
        self.gamma        = gamma
        self.lam          = lam
        self.n_epochs     = n_epochs
        self.loss_fn      = Loss_PPO(cv=cv, ce=ce,
                                    clip_epsilon=clip_epsilon)

    def compute_gae(self, deltas: list, # warning : this methode is only with ppo used 
                    dones:list, # otherwise delete this one 
                    gamma: float = None,
                    lam:   float = None) -> torch.Tensor:
        gamma = gamma if gamma is not None else self.gamma
        lam   = lam   if lam   is not None else self.lam
        T     = len(deltas)
        advantages = torch.zeros(T)
        gae        = torch.tensor(0.0)
        for t in reversed(range(T)):
            if dones[t]:
                gae = torch.tensor(0.0)
            gae           = deltas[t] + gamma * lam * gae
            advantages[t] = gae.detach()
        return advantages

    def update(self, policy: Policy_PPO, rollout: dict) -> float:

        deltas         = rollout["deltas"]
        states_visited = rollout["states_visited"]
        log_probs_old  = rollout["log_probs_old"]
        actions_taken  = rollout["actions_taken"]
        values_visited = rollout["values_visited"]
        dones = rollout["dones"] # added to fix the leaking

        A        = self.compute_gae(deltas, dones) # added dones to fix the leaking
        v_tensor = torch.stack([v.detach() for v in values_visited])
        returns  = (A + v_tensor).detach()

        #normalize advantages — stabilizes training
        #if torch.isfinite(A).all() and A.std() > 1e-8: #I added the torch.isfinite to solve the NaN issue 
        if torch.isfinite(A.std()) and A.std() > 1e-8:
            A = (A - A.mean()) / (A.std() + 1e-8)
        
        #new condition added :
        else :
            A = A - A.mean() # at least center even if std is too small

        total_loss = 0.0

        for _ in range(self.n_epochs):
            log_probs_new, values_new, probs_all = [], [], []

            for s, a in zip(states_visited, actions_taken):
                x        = torch.as_tensor(policy.state_to_vector_fn(s), dtype=torch.float32)
                probs, v = policy.net(x)
                dist     = torch.distributions.Categorical(probs)
                log_p    = dist.log_prob(torch.tensor(policy.actions.index(a)))
                log_probs_new.append(log_p)
                values_new.append(v)
                probs_all.append(probs)

            log_probs_new = torch.stack(log_probs_new)
            log_probs_old_t = torch.stack(log_probs_old)
            v_pred        = torch.stack(values_new)
            probs_tensor  = torch.stack(probs_all)

            loss = self.loss_fn(log_probs_new, log_probs_old_t,
                                A, v_pred, returns, probs_tensor)

            self.optimizer.zero_grad()
            loss.backward()

            # replace the single line :
            #torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
            
            # through these twos
            torch.nn.utils.clip_grad_norm_(
                list(policy.net.body.parameters()) +
                list(policy.net.actor.parameters()),
                max_norm=0.5
            )
            torch.nn.utils.clip_grad_norm_(
                policy.net.critic.parameters(),
                max_norm=0.5
            )

            self.optimizer.step()
            total_loss += loss.item()

        return total_loss / self.n_epochs

    def snapshot(self) -> dict:
        return {
            "gamma":        self.gamma,
            "lam":          self.lam,
            "clip_epsilon": self.clip_epsilon,
            "cv":           self.cv,
            "ce":           self.ce,
            "lr":           self.optimizer.param_groups[0]["lr"],
    }

## new ppo class for both discrete and continous 


In [65]:
class PPO(LearningStrategy):
    def __init__(
        self,
        optimizer,
        clip_epsilon: float = 0.2,
        cv: float = 0.5,
        ce: float = 0.01,
        gamma: float = 0.99,
        lam: float = 0.95,
        n_epochs: int = 1
    ):
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon
        self.cv = cv
        self.ce = ce
        self.gamma = gamma
        self.lam = lam
        self.n_epochs = n_epochs
        self.loss_fn = Loss_PPO(cv=cv, ce=ce, clip_epsilon=clip_epsilon)

    def compute_gae(self, deltas, dones, gamma=None, lam=None):
        gamma = gamma if gamma is not None else self.gamma
        lam = lam if lam is not None else self.lam
        T = len(deltas)
        advantages = torch.zeros(T)
        gae = torch.tensor(0.0)

        for t in reversed(range(T)):
            if dones[t]:
                gae = torch.tensor(0.0)
            gae = deltas[t] + gamma * lam * gae
            advantages[t] = gae.detach()
        return advantages

    def update(self, policy, rollout):
        if policy.net.action_mode == "discrete":
            return self._update_discrete(policy, rollout)
        elif policy.net.action_mode == "continuous":
            return self._update_continuous(policy, rollout)
        else:
            raise ValueError(f"Unknown action mode: {policy.net.action_mode}")

    def _update_discrete(self, policy, rollout):
        return self._update_common(policy, rollout, continuous=False)

    def _update_continuous(self, policy, rollout):
        return self._update_common(policy, rollout, continuous=True)

    def _update_common(self, policy, rollout, continuous: bool):
        deltas = rollout["deltas"]
        states_visited = rollout["states_visited"]
        log_probs_old = rollout["log_probs_old"]
        actions_taken = rollout["actions_taken"]
        values_visited = rollout["values_visited"]
        dones = rollout["dones"]

        # ----- Advantages & Returns -----
        A = self.compute_gae(deltas, dones)
        v_tensor = torch.stack([v.detach() for v in values_visited])
        returns = (A + v_tensor).detach()

        # Normalise advantages
        if torch.isfinite(A.std()) and A.std() > 1e-8:
            A = (A - A.mean()) / (A.std() + 1e-8)
        else:
            A = A - A.mean()

        total_loss = 0.0

        for _ in range(self.n_epochs):
            log_probs_new = []
            values_new = []
            entropies = []

            for s, a in zip(states_visited, actions_taken):
                x = torch.as_tensor(policy.state_to_vector_fn(s),dtype=torch.float32)
                dist, value = policy.net(x)

                if continuous:
                    action = torch.as_tensor(a, dtype=torch.float32)
                    log_prob = dist.log_prob(action).sum()
                    entropy = dist.entropy().sum()
                else:
                    action_idx = policy.actions.index(a)
                    log_prob = dist.log_prob(torch.tensor(action_idx))
                    entropy = dist.entropy()

                log_probs_new.append(log_prob)
                values_new.append(value)
                entropies.append(entropy)

            log_probs_new = torch.stack(log_probs_new)
            log_probs_old_t = torch.stack(log_probs_old)
            v_pred = torch.stack(values_new)
            entropy = torch.stack(entropies).mean()

            loss = self.loss_fn(
                log_probs_new,
                log_probs_old_t,
                A,
                v_pred,
                returns,
                entropy
            )

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / self.n_epochs

    def snapshot(self):
        return {
            "gamma": self.gamma,
            "lam": self.lam,
            "clip_epsilon": self.clip_epsilon,
            "cv": self.cv,
            "ce": self.ce,
            "lr": self.optimizer.param_groups[0]["lr"],
        }

In [66]:
def debug_ppo(env, agent, rollout=None):

    print("\n" + "=" * 70)
    print("PPO DEBUG")
    print("=" * 70)

    ############################################################
    # 1. ENVIRONMENT
    ############################################################

    print("\n[1] ENVIRONMENT")
    print("-" * 70)

    for s in env.states:

        for action in agent.policy.actions:

            old = env.current
            env.current = s

            exp = env.step(action)

            env.current = old

            ns = exp["next_state"]

            print(
                f"{s.name:>3} -- {action:<6}"
                f" --> {ns.name:<3}"
                f" reward={exp['reward']:>6}"
                f" done={exp['done']}"
                f" symbol={ns.symbol}"
            )

    ############################################################
    # 2. CRITIC
    ############################################################

    print("\n" + "=" * 70)
    print("[2] VALUE FUNCTION")
    print("=" * 70)

    for s in env.states:

        x = torch.as_tensor(
            agent.policy.state_to_vector_fn(s),
            dtype=torch.float32
        )

        with torch.no_grad():
            _, value = agent.policy.net(x)

        print(
            f"{s.name:<3}"
            f" symbol={str(s.symbol):<5}"
            f" value={value.item():10.3f}"
        )

    ############################################################
    # 3. ACTOR
    ############################################################

    print("\n" + "=" * 70)
    print("[3] POLICY")
    print("=" * 70)

    mode = agent.policy.net.action_mode

    print(f"Action mode: {mode}")

    for s in env.states:

        x = torch.as_tensor(
            agent.policy.state_to_vector_fn(s),
            dtype=torch.float32
        )

        with torch.no_grad():
            dist, _ = agent.policy.net(x)

        print(f"\n{s.name}")

        if mode == "discrete":

            probs = dist.probs

            for a, p in zip(
                agent.policy.actions,
                probs
            ):
                print(
                    f"   {a:<6}: {p.item():.4f}"
                )

            best = torch.argmax(probs).item()

            print(
                f"   best action : "
                f"{agent.policy.actions[best]}"
            )

            print(
                f"   entropy     : "
                f"{dist.entropy().item():.4f}"
            )

        elif mode == "continuous":

            mean = dist.mean
            std = dist.stddev

            print(
                f"   mean : {mean.detach().numpy()}"
            )

            print(
                f"   std  : {std.detach().numpy()}"
            )

            print(
                f"   entropy : "
                f"{dist.entropy().sum().item():.4f}"
            )

    ############################################################
    # 4. ROLLOUT
    ############################################################

    if rollout is not None:

        print("\n" + "=" * 70)
        print("[4] ROLLOUT")
        print("=" * 70)

        A = agent.strategy.compute_gae(
            rollout["deltas"],
            rollout["dones"]
        )

        values = torch.stack(
            rollout["values_visited"]
        )

        returns = A + values

        for i in range(
            len(rollout["states_visited"])
        ):

            state = rollout["states_visited"][i]

            print(
                f"\nstep {i:02d}"
                f"  state={state.name}"
                f"  action={rollout['actions_taken'][i]}"
                f"  done={rollout['dones'][i]}"
            )

            print(
                f"   value      : "
                f"{values[i].item():10.3f}"
            )

            print(
                f"   delta      : "
                f"{rollout['deltas'][i].item():10.3f}"
            )

            print(
                f"   advantage  : "
                f"{A[i].item():10.3f}"
            )

            print(
                f"   return     : "
                f"{returns[i].item():10.3f}"
            )

            if "log_probs_old" in rollout:

                print(
                    f"   old log_p  : "
                    f"{rollout['log_probs_old'][i].item():10.4f}"
                )

    ############################################################
    # 5. PARAMETERS
    ############################################################

    print("\n" + "=" * 70)
    print("[5] PARAMETERS")
    print("=" * 70)

    for name, p in agent.policy.net.named_parameters():

        print(f"\n{name}")

        print(
            f" mean : {p.data.mean().item():.6f}"
        )

        print(
            f" std  : {p.data.std().item():.6f}"
        )

        print(
            f" min  : {p.data.min().item():.6f}"
        )

        print(
            f" max  : {p.data.max().item():.6f}"
        )

        print(
            f" nan  : "
            f"{torch.isnan(p.data).any().item()}"
        )

        print(
            f" inf  : "
            f"{torch.isinf(p.data).any().item()}"
        )

    ############################################################
    # 6. GRADIENTS
    ############################################################

    print("\n" + "=" * 70)
    print("[6] GRADIENTS")
    print("=" * 70)

    for name, p in agent.policy.net.named_parameters():

        if p.grad is None:

            print(f"\n{name}: NO GRADIENT")
            continue

        print(f"\n{name}")

        print(
            f" norm : {p.grad.norm().item():.6f}"
        )

        print(
            f" mean : {p.grad.mean().item():.6f}"
        )

        print(
            f" std  : {p.grad.std().item():.6f}"
        )

        print(
            f" nan  : "
            f"{torch.isnan(p.grad).any().item()}"
        )

        print(
            f" inf  : "
            f"{torch.isinf(p.grad).any().item()}"
        )

    print("\n" + "=" * 70)
    print("END DEBUG")
    print("=" * 70)

In [67]:
def debug_learning_balance(policy):
    """
    Compare actor and critic gradient magnitudes.

    Call AFTER loss.backward()
    and BEFORE optimizer.step().

    This is a diagnostic, not a measure of
    whether PPO is mathematically balanced.
    """

    actor_sq = 0.0
    critic_sq = 0.0

    for name, param in policy.net.named_parameters():

        if param.grad is None:
            continue

        g = param.grad.detach().norm().item()

        if "actor" in name:
            actor_sq += g ** 2

        elif "critic" in name:
            critic_sq += g ** 2

    actor_norm = actor_sq ** 0.5
    critic_norm = critic_sq ** 0.5

    ratio = actor_norm / (critic_norm + 1e-8)

    print("=" * 60)
    print("LEARNING BALANCE")
    print("=" * 60)

    print(f"Actor gradient : {actor_norm:.6f}")
    print(f"Critic gradient: {critic_norm:.6f}")
    print(f"Ratio A/C      : {ratio:.3f}")

    if actor_norm < 1e-8 and critic_norm < 1e-8:
        status = "No significant gradients"

    elif actor_norm < 1e-8:
        status = "Actor has no gradient"

    elif critic_norm < 1e-8:
        status = "Critic has no gradient"

    elif ratio < 0.1:
        status = "Critic gradient much larger"

    elif ratio > 10.0:
        status = "Actor gradient much larger"

    else:
        status = "No major imbalance detected"

    print(f"Status : {status}")

    return {
        "actor": actor_norm,
        "critic": critic_norm,
        "ratio": ratio
    }

## Create the Agent

In [68]:
# ════════════════════════════════════════════════════════════════════
#  BRICK 4 — AGENT
#  The self. Owns the policy (permanent). Borrows the strategy (swappable).
#  Contains no learning logic — purely coordinates the other bricks.
# ════════════════════════════════════════════════════════════════════
    
class Agent:

    def __init__(self, strategy: LearningStrategy, policy: Policy):
        self.policy   = policy    # permanent — never replaced
        self.strategy = strategy  # swappable — plug any algorithm in
        self.memory   = []        # full experience trace

    def act(self, state) -> str:
        """Ask the strategy what to do, passing the policy as context."""
        return self.policy.select_action(state)

    def learn(self, experience:dict): # experience = {"state": "A","action": "right","reward": 1,"next_state": "B","done": False}
        """Tell the strategy what happened; it writes into the policy."""
        

        self.strategy.update(self.policy, experience)
        self.memory.append((
            experience["state"].name, 
            experience["action"], 
            experience["reward"], 
            experience["next_state"].name, 
            experience["done"]
        ))

    def swap_strategy(self, new_strategy: LearningStrategy):
        """
        Replace the learning algorithm.
        The policy — and everything it has learned — is untouched.
        """
        self.strategy = new_strategy

In [69]:

class Agent_PPO(Agent):

    def __init__(self, strategy: PPO, policy: Policy_PPO):
        super().__init__(strategy, policy)

    def act(self, state):
        return self.policy.select_action(state)

    def learn(self, rollout):
        loss = self.strategy.update(self.policy, rollout)
        self.memory.append({
            "rollout_size": len(rollout["deltas"]), 
            "loss":loss
        })

    def value(self, state):
        x = torch.as_tensor(
            self.policy.state_to_vector_fn(state),
            dtype=torch.float32
        )

        _, value = self.policy.net(x)
        return value

## Training function 

In [70]:
def randomize_board(env, n_goals=3, n_traps=10):
    for s in env.states:
        s.symbol = None

    # copy + exclude start so it's never a trap or goal
    blocks = [s for s in env.states if s != env.start]
    random.shuffle(blocks)

    n_goals = min(n_goals, len(blocks))
    n_traps = min(n_traps, len(blocks) - n_goals)

    goal_cells   = blocks[:n_goals]
    trap_cells   = blocks[n_goals: n_goals + n_traps]
    normal_cells = blocks[n_goals + n_traps:]

    for c in goal_cells: c.symbol = "O"
    for c in trap_cells: c.symbol = "X"

    return goal_cells, trap_cells, normal_cells



In [71]:
def randomize_board_blocked(env, n_goals=1, n_traps=5):

    # Clear the board
    for s in env.states:
        s.symbol = None

    # Start cannot be goal or trap
    free = [
        s for s in env.states
        if s != env.start
    ]

    random.shuffle(free)

    # --------------------------------------------------------
    # PLACE GOALS
    # --------------------------------------------------------

    n_goals = min(n_goals, len(free))

    goal_cells = free[:n_goals]

    for goal in goal_cells:
        goal.symbol = "O"

    # --------------------------------------------------------
    # FIND CELLS BETWEEN START AND GOALS
    # --------------------------------------------------------

    sx, sy = env.start.coordinate

    candidate_cells = []

    for cell in free[n_goals:]:

        cx, cy = cell.coordinate

        # Check whether the cell lies approximately
        # between start and at least one goal
        for goal in goal_cells:

            gx, gy = goal.coordinate

            between_x = (
                min(sx, gx)
                <= cx
                <= max(sx, gx)
            )

            between_y = (
                min(sy, gy)
                <= cy
                <= max(sy, gy)
            )

            if between_x or between_y:

                candidate_cells.append(cell)

                break

    # --------------------------------------------------------
    # PLACE SOME TRAPS AS OBSTACLES
    # --------------------------------------------------------

    random.shuffle(candidate_cells)

    path_traps = min(
        n_traps // 2,
        len(candidate_cells)
    )

    trap_cells = candidate_cells[:path_traps]

    for cell in trap_cells:
        cell.symbol = "X"

    # --------------------------------------------------------
    # PLACE REMAINING TRAPS RANDOMLY
    # --------------------------------------------------------

    remaining_cells = [
        cell
        for cell in free[n_goals:]
        if cell.symbol is None
    ]

    random.shuffle(remaining_cells)

    remaining_traps = min(
        n_traps - path_traps,
        len(remaining_cells)
    )

    extra_traps = remaining_cells[
        :remaining_traps
    ]

    for cell in extra_traps:
        cell.symbol = "X"

    # Complete trap list
    trap_cells += extra_traps

    # --------------------------------------------------------
    # NORMAL CELLS
    # --------------------------------------------------------

    normal_cells = [
        cell
        for cell in env.states
        if cell.symbol is None
    ]

    return goal_cells, trap_cells, normal_cells

In [72]:
def training(
    env: Env,
    agent: Agent_PPO,
    episodes: int = 100,
    steps: int = 50,
    rollout_length: int = 256,
    random_start: bool = False
):
    """
    PPO training.

    IMPORTANT:

    Episode ending != PPO update.

    The rollout buffer continues across episodes.

    PPO updates ONLY when enough experience has been collected.
    """

    state_to_vector_fn = agent.policy.state_to_vector_fn


    # =========================================================
    # PPO ROLLOUT BUFFER
    # =========================================================

    deltas = []
    log_probs_old = []
    states_visited = []
    actions_taken = []
    values_visited = []
    dones = []


    # =========================================================
    # HELPER FUNCTION
    # =========================================================

    def update_from_buffer():

        nonlocal deltas
        nonlocal log_probs_old
        nonlocal states_visited
        nonlocal actions_taken
        nonlocal values_visited
        nonlocal dones

        if len(deltas) == 0:
            return

        agent.learn({

            "deltas": deltas,

            "states_visited": states_visited,

            "log_probs_old": log_probs_old,

            "actions_taken": actions_taken,

            "values_visited": values_visited,

            "dones": dones

        })


        # Clear buffer AFTER PPO update

        deltas.clear()

        log_probs_old.clear()

        states_visited.clear()

        actions_taken.clear()

        values_visited.clear()

        dones.clear()


    # =========================================================
    # TRAINING EPISODES
    # =========================================================

    for episode in range(episodes):


        # =====================================================
        # RANDOM START
        # =====================================================

        free_states = [

            s for s in env.states

            if s.symbol not in ("X", "O")

        ]


        trap_neighbours = [

            s for s in free_states

            if any(
                n is not None and n.symbol == "X"
                for n in s.neighbours.values()
            )

        ]


        start_pool = (

            free_states

            +

            trap_neighbours * 3

        )


        if random_start and start_pool:

            env.current = random.choice(start_pool)

        else:

            env.reset()


        s = env.current


        # =====================================================
        # EPISODE LOOP
        # =====================================================

        for step in range(steps):


            # -------------------------------------------------
            # ACT
            # -------------------------------------------------

            a, log_prob, v_s = agent.act(s)


            # -------------------------------------------------
            # ENVIRONMENT STEP
            # -------------------------------------------------

            exp = env.step(a)

            n_s = exp["next_state"]

            r = exp["reward"]

            done = exp["done"]


            # -------------------------------------------------
            # NORMALIZE REWARD
            # -------------------------------------------------

            r = r / 10.0


            # -------------------------------------------------
            # NEXT STATE VALUE
            # -------------------------------------------------

            with torch.no_grad():

                if done:

                    v_ns = torch.tensor(0.0)

                else:

                    x = torch.as_tensor(

                        state_to_vector_fn(n_s),

                        dtype=torch.float32

                    )

                    _, v_ns = agent.policy.net(x)

                    v_ns = v_ns.squeeze()


            # -------------------------------------------------
            # TD ERROR
            #
            # δ = r + γV(s') - V(s)
            # -------------------------------------------------

            delta = (

                r

                +

                agent.strategy.gamma * v_ns

                -

                v_s.detach()

            )


            # -------------------------------------------------
            # STORE EXPERIENCE
            # -------------------------------------------------

            deltas.append(delta)

            states_visited.append(s)

            actions_taken.append(a)

            values_visited.append(v_s.detach())

            log_probs_old.append(log_prob.detach())

            dones.append(done)


            # -------------------------------------------------
            # MOVE
            # -------------------------------------------------

            s = n_s


            # =================================================
            # PPO UPDATE ONLY WHEN BUFFER IS FULL
            # =================================================

            if len(deltas) >= rollout_length:

                update_from_buffer()


            # =================================================
            # EPISODE ENDS
            # =================================================

            if done:

                break


    # =========================================================
    # FINAL UPDATE
    #
    # Update remaining samples after ALL episodes are finished.
    # =========================================================

    if len(deltas) > 0:

        update_from_buffer()

In [73]:
import copy

def training_set_generator_old(configs, shuffle=True):
    """
    configs: list of dicts, each with:
        based_env    : Chessboard to copy from
        n_goals      : int
        n_traps      : int
        number_of_sets: int

    Example:
        configs = [
            {"based_env": env,  "n_goals": 1, "n_traps": 0, "number_of_sets": 20},
            {"based_env": env,  "n_goals": 1, "n_traps": 2, "number_of_sets": 30},
            {"based_env": env1, "n_goals": 3, "n_traps": 10,"number_of_sets": 50},
        ]
    """
    # backward-compatible: accept old single-env call signature
    if isinstance(configs, Env):
        raise TypeError(
            "training_set_generator now takes a list of config dicts. "
            "See docstring for format."
        )

    training_set = []

    for cfg in configs:
        based_env      = cfg["based_env"]
        n_goals        = cfg["n_goals"]
        n_traps        = cfg["n_traps"]
        number_of_sets = cfg["number_of_sets"]

        for _ in range(number_of_sets):
            env_copy = copy.deepcopy(based_env)
            randomize_board(env_copy, n_goals=n_goals, n_traps=n_traps)
            training_set.append(env_copy)

    if shuffle:
        random.shuffle(training_set)

    print(f"Generated {len(training_set)} environments from {len(configs)} configs")
    return training_set


def test_agent(agent, test_set, max_steps=50, n_episodes=10):
    results = []

    for i, test_env in enumerate(test_set):
        print(f"\n===== Testing environment {i+1}/{len(test_set)} =====")

        goal_cells   = [c for c in test_env.states if c.symbol == "O"]
        trap_cells   = [c for c in test_env.states if c.symbol == "X"]
        normal_cells = [c for c in test_env.states if c.symbol is None]
        print(f"Goals  ({len(goal_cells)}): {[c.name for c in goal_cells]}")
        print(f"Traps  ({len(trap_cells)}): {[c.name for c in trap_cells]}")
        print(f"Normal ({len(normal_cells)}): {len(normal_cells)} cells")

        successes = 0
        traps_hit = 0
        timeouts  = 0

        for episode in range(n_episodes):
            test_env.reset()
            Path = []

            for step in range(max_steps):
                s          = test_env.current
                a, _, _    = agent.act(s)
                exp        = test_env.step(a)
                Path.append(s)

                if exp["done"]:
                    if exp["next_state"].symbol == "O":
                        successes += 1
                        print(f"  ep {episode+1}: GOAL in {step+1} steps")
                        Path.append(exp["next_state"])
                    else:
                        traps_hit += 1
                        print(f"  ep {episode+1}: TRAP in {step+1} steps")
                        Path.append(exp["next_state"])
                    break
            else:
                timeouts += 1
                print(f"  ep {episode+1}: TIMEOUT")
            
            print(f"here is the full path in this episode : {[c.name for c in Path]}")

        env_result = {
            "goals": successes, "traps": traps_hit, "timeouts": timeouts,
            "success_rate": successes / n_episodes
        }
        results.append(env_result)
        print(f"  → {successes}/{n_episodes} goals  "
              f"{traps_hit} traps  {timeouts} timeouts")

    overall = sum(r["goals"] for r in results) / (len(results) * n_episodes)
    print(f"\n{'='*50}")
    print(f"OVERALL SUCCESS RATE: {overall:.1%} across {len(test_set)} envs")
    return results

In [74]:
def training_set_generator(configs, shuffle=True):

    training_set = []

    for cfg in configs:

        based_env      = cfg["based_env"]
        n_goals        = cfg["n_goals"]
        n_traps        = cfg["n_traps"]
        number_of_sets = cfg["number_of_sets"]

        # If "type" is not mentioned → use normal random generation
        env_type = cfg.get("type", "random")

        for _ in range(number_of_sets):

            env_copy = copy.deepcopy(based_env)

            if env_type == "random":

                randomize_board(
                    env_copy,
                    n_goals=n_goals,
                    n_traps=n_traps
                )

            elif env_type == "blocked":

                randomize_board_blocked(
                    env_copy,
                    n_goals=n_goals,
                    n_traps=n_traps
                )

            else:
                raise ValueError(
                    f"Unknown environment type: {env_type}"
                )

            training_set.append(env_copy)

    if shuffle:
        random.shuffle(training_set)

    print(
        f"Generated {len(training_set)} environments "
        f"from {len(configs)} configs"
    )

    return training_set

In [75]:



import copy
import random
import itertools

# ══════════════════════════════════════════════════════
# LEVEL 1 — BASIS: one piece from a spec
# ══════════════════════════════════════════════════════

def make_basis_piece(spec: dict, piece_id: str) -> Board:

    geometry = spec["geometry"]

    if geometry == "box":
        N, M  = spec["shape"]
        cells = [
            CellBox(f"{piece_id}_c{i}", (i % N, i // N), "white",
                    None, [None, None, None, None])
            for i in range(N * M)
        ]
        return Board(piece_id, cells, geometry="box", data={"shape": (N, M)})

    elif geometry in ("h_line", "v_line"):
        L     = spec["length"]
        cells = [
            CellBox(f"{piece_id}_c{i}",
                    (i, 0) if geometry == "h_line" else (0, i),
                    "white", None, [None, None, None, None])
            for i in range(L)
        ]
        return Board(piece_id, cells, geometry=geometry, data={})

    else:
        raise ValueError(f"Unknown geometry: {geometry}")


# ══════════════════════════════════════════════════════
# LEVEL 2 — COMPOSITION: connect pieces into one board
# ══════════════════════════════════════════════════════

def compose_pieces(pieces: list, piece_id: str = "composite") -> Board:

    all_cells = []
    for board in pieces:
        all_cells.extend(board.cells)

    coord_map = {c.coordinate: c for c in all_cells}

    directions = {
        "up":    (0,  1),
        "down":  (0, -1),
        "right": (1,  0),
        "left":  (-1, 0),
    }
    for cell in all_cells:
        x, y = cell.coordinate
        for direction, (dx, dy) in directions.items():
            neighbour = coord_map.get((x + dx, y + dy))
            if neighbour is not None:
                cell.neighbours[direction] = neighbour

    # build a Board without re-running form() since neighbours are already wired
    composite      = Board.__new__(Board)
    composite.id   = piece_id
    composite.cells = all_cells
    composite.geometry = "custom"
    composite.data = None

    for c in all_cells:
        c.board = composite

    return composite


# ══════════════════════════════════════════════════════
# LEVEL 3 — ENVIRONMENT DATASET
# ══════════════════════════════════════════════════════

def safe_randomize(env, n_goals: int, n_traps: int):
    """
    Randomize goals and traps, never touching the start cell.
    """
    start = env.start

    # reset all non-start cells
    for s in env.states:
        if s != start:
            s.symbol = None

    candidates = [s for s in env.states if s != start]
    random.shuffle(candidates)

    # guard: can't place more goals+traps than free cells
    total = len(candidates)
    n_goals = min(n_goals, total)
    n_traps = min(n_traps, total - n_goals)

    for c in candidates[:n_goals]:
        c.symbol = "O"
    for c in candidates[n_goals: n_goals + n_traps]:
        c.symbol = "X"


class EnvironmentGenerator:
    """
    3-level environment generator.

    Usage:
        generator = EnvironmentGenerator(seed=42)
        envs = generator.generate(
            configs=[
                {"specs": [{"geometry":"box","shape":(3,3)}],
                 "n_pieces": 1, "n_goals": 1, "n_traps": 1,  "n_envs": 30},
                {"specs": [{"geometry":"box","shape":(3,3)},
                           {"geometry":"box","shape":(3,3)}],
                 "n_pieces": 2, "n_goals": 2, "n_traps": 5,  "n_envs": 50},
                {"specs": [{"geometry":"box","shape":(9,9)}],
                 "n_pieces": 1, "n_goals": 3, "n_traps": 12, "n_envs": 70},
            ],
            actions=Actions,
            shuffle=True
        )
    """

    def __init__(self, seed: int = None):
        if seed is not None:
            random.seed(seed)
        self._counter = 0

    def _make_env(self, specs: list, n_pieces: int,
                  n_goals: int, n_traps: int,
                  actions: list) -> "Chessboard":
        """Build one environment from randomly chosen specs."""

        self._counter += 1

        # pick n_pieces specs (with replacement if fewer specs than pieces)
        chosen = [random.choice(specs) for _ in range(n_pieces)]

        # build pieces with offset coordinates so they don't overlap
        pieces  = []
        x_offset = 0
        for i, spec in enumerate(chosen):
            pid   = f"env{self._counter}_p{i}"
            board = make_basis_piece(spec, pid)

            # shift x coordinates to avoid overlap
            for c in board.cells:
                cx, cy = c.coordinate
                c.coordinate = (cx + x_offset, cy)

            x_offset += max(c.coordinate[0] for c in board.cells) + 2
            pieces.append(board)

        composite = compose_pieces(pieces, f"env{self._counter}")

        # pick a free start cell
        start = random.choice(composite.cells)

        env = Chessboard(board=composite, actions=actions, start=start)
        safe_randomize(env, n_goals=n_goals, n_traps=n_traps)

        return env

    def generate(self,
                 configs:  list,
                 actions:  list,
                 shuffle:  bool = True) -> list:
        """
        configs: list of dicts, each with:
            specs    : list of geometry specs to draw pieces from
            n_pieces : how many pieces to compose per env
            n_goals  : goals per env
            n_traps  : traps per env
            n_envs   : how many envs of this config to generate
        """
        all_envs = []

        for cfg in configs:
            specs    = cfg["specs"]
            n_pieces = cfg.get("n_pieces", 1)
            n_goals  = cfg["n_goals"]
            n_traps  = cfg["n_traps"]
            n_envs   = cfg["n_envs"]

            for _ in range(n_envs):
                env = self._make_env(
                    specs    = specs,
                    n_pieces = n_pieces,
                    n_goals  = n_goals,
                    n_traps  = n_traps,
                    actions  = actions
                )
                all_envs.append(env)

        if shuffle:
            random.shuffle(all_envs)

        print(f"Generated {len(all_envs)} environments "
              f"from {len(configs)} configs")
        return all_envs

In [76]:
def training_multi(
    env_list,
    agent,
    total_episodes: int = 8000,
    steps: int = 50,
    rollout_length: int = 256,
    random_start: bool = True
):

    state_to_vector_fn = agent.policy.state_to_vector_fn


    # =========================================================
    # SHARED PPO BUFFER
    # =========================================================

    deltas = []

    log_probs_old = []

    states_visited = []

    actions_taken = []

    values_visited = []

    dones = []


    # =========================================================
    # UPDATE FUNCTION
    # =========================================================

    def update_from_buffer():

        nonlocal deltas
        nonlocal log_probs_old
        nonlocal states_visited
        nonlocal actions_taken
        nonlocal values_visited
        nonlocal dones


        if len(deltas) == 0:

            return


        agent.learn({

            "deltas": deltas,

            "states_visited": states_visited,

            "log_probs_old": log_probs_old,

            "actions_taken": actions_taken,

            "values_visited": values_visited,

            "dones": dones

        })


        deltas.clear()

        log_probs_old.clear()

        states_visited.clear()

        actions_taken.clear()

        values_visited.clear()

        dones.clear()


    # =========================================================
    # EPISODES
    # =========================================================

    for episode in range(total_episodes):


        # RANDOM ENVIRONMENT

        env = random.choice(env_list)


        # -----------------------------------------------------
        # RANDOM START
        # -----------------------------------------------------

        free_states = [

            s for s in env.states

            if s.symbol not in ("X", "O")

        ]


        if random_start and free_states:

            env.current = random.choice(free_states)

        else:

            env.reset()


        s = env.current


        # =====================================================
        # EPISODE
        # =====================================================

        for step in range(steps):


            # ACTION

            a, log_prob, v_s = agent.act(s)


            # ENVIRONMENT

            exp = env.step(a)

            n_s = exp["next_state"]

            r = float(exp["reward"])

            done = exp["done"]


            # NEXT VALUE

            with torch.no_grad():

                if done:

                    v_ns = torch.tensor(0.0)

                else:

                    x = torch.as_tensor(

                        state_to_vector_fn(n_s),

                        dtype=torch.float32

                    )

                    _, v_ns = agent.policy.net(x)

                    v_ns = v_ns.squeeze()


            # TD ERROR

            delta = (

                r

                +

                agent.strategy.gamma * v_ns

                -

                v_s.detach()

            )


            # STORE

            deltas.append(delta)

            states_visited.append(s)

            actions_taken.append(a)

            values_visited.append(v_s.detach())

            log_probs_old.append(log_prob.detach())

            dones.append(done)


            s = n_s


            # PPO UPDATE

            if len(deltas) >= rollout_length:

                update_from_buffer()


            # EPISODE ENDS

            if done:

                break


    # =========================================================
    # FINAL UPDATE
    # =========================================================

    if len(deltas) > 0:

        update_from_buffer()

In [77]:
# comment or uncomment the ones you want to use or remove

# Classic
#env0 = Chessboard(board=claude_board, actions=Actions, start=s1)

# Stronger reward
env0 = Chessboard(board=claude_board, actions=Actions, start=s1,reward_fn= reward_v4)

# Distance shaping
#env0 = Chessboard(board=claude_board, actions=Actions, start=s1,reward_fn= reward_v3)

In [ ]:
# ── usage ────────────────────────────────────────────────────────────────
n_features = len(state_to_vector(env0.states[0]))    
action_dim  = len(Actions)                           

net       = ActorCritic(n_features=n_features, action_dim=action_dim, hidden_dim=64)
pi        = Policy_PPO(net ,Actions)
optimizer = torch.optim.Adam(net.parameters(), lr=3e-4)
ppo       = PPO(
    optimizer=optimizer, 
    clip_epsilon=0.2,
      gamma=0.9, lam=0.95 ,
      cv = 0.5 , ce= 0.06, # decrease ce to 0.06
      n_epochs=6 ) # lam switch from 0.95 to 0.80
agent     = Agent_PPO(strategy=ppo, policy=pi)

#env = Chessboard(board=claude_board, actions=Actions, start=s1)
training_envs_0 = training_set_generator([
    {"based_env": env0, "n_goals": 1, "n_traps": 0, "number_of_sets": 15}, #15
    {"based_env": env0, "n_goals": 1, "n_traps": 1, "number_of_sets": 9*9 , "type":"blocked"}, #25
    {"based_env": env0, "n_goals": 2, "n_traps": 2, "number_of_sets": 100}, #10
])

for n,env_0 in enumerate(training_envs_0):

     print(f"currently training in env n°:{n+1}/{len(training_envs_0)}")
    
     training(env=env_0, agent=agent, episodes=500 , steps = 500 ,rollout_length= 256 , # rollout_length = 64*4
             random_start=True )
print("training completed")

# training_multi(

#     env_list=training_envs_0,

#     agent=agent,

#     total_episodes=10000,

#     steps=50,

#     rollout_length=256,

#     random_start=True

# )

# print("training completed")


Generated 196 environments from 3 configs
currently training in env n°:1/196
currently training in env n°:2/196
currently training in env n°:3/196
currently training in env n°:4/196
currently training in env n°:5/196
currently training in env n°:6/196
currently training in env n°:7/196
currently training in env n°:8/196
currently training in env n°:9/196
currently training in env n°:10/196
currently training in env n°:11/196
currently training in env n°:12/196
currently training in env n°:13/196
currently training in env n°:14/196
currently training in env n°:15/196
currently training in env n°:16/196
currently training in env n°:17/196
currently training in env n°:18/196
currently training in env n°:19/196
currently training in env n°:20/196
currently training in env n°:21/196
currently training in env n°:22/196
currently training in env n°:23/196
currently training in env n°:24/196
currently training in env n°:25/196
currently training in env n°:26/196
currently training in env n°:27

In [ ]:
debug_ppo(agent = agent,env = env0)
debug_learning_balance(agent.policy)



PPO COMPLETE DEBUG

[1] ENVIRONMENT
----------------------------------------------------------------------
 s1 -- up     --> s4  reward=0.09000000000000001 done=False symbol=None
 s1 -- right  --> s2  reward=0.09000000000000001 done=False symbol=None
 s1 -- down   --> s1  reward= -0.01 done=False symbol=None
 s1 -- left   --> s1  reward= -0.01 done=False symbol=None
 s2 -- up     --> s5  reward=  -1.0 done=True symbol=X
 s2 -- right  --> s3  reward=0.09000000000000001 done=False symbol=None
 s2 -- down   --> s2  reward= -0.01 done=False symbol=None
 s2 -- left   --> s1  reward= -0.11 done=False symbol=None
 s3 -- up     --> s6  reward=0.09000000000000001 done=False symbol=None
 s3 -- right  --> s3  reward= -0.01 done=False symbol=None
 s3 -- down   --> s3  reward= -0.01 done=False symbol=None
 s3 -- left   --> s2  reward= -0.11 done=False symbol=None
 s4 -- up     --> s7  reward=0.09000000000000001 done=False symbol=None
 s4 -- right  --> s5  reward=  -1.0 done=True symbol=X
 s4 -- do

/tmp/ipykernel_43626/275698226.py:132: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.data.std().item():.6f}")
/tmp/ipykernel_43626/275698226.py:154: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.grad.std().item():.6f}")


{'actor': 0.03566507099252652,
 'critic': 0.04060677707539226,
 'ratio': 0.8783031990762838}

In [ ]:
#env = Chessboard(board=claude_board, actions=Actions, start=s1)

test_set_0 = training_set_generator(
    configs= [{"based_env": env0,  "n_goals": 1, "n_traps": 1, "number_of_sets": 50,"type":"blocked"}]
)

Generated 50 environments from 1 configs


In [ ]:

results = test_agent(agent, test_set_0, max_steps=500, n_episodes=15)


===== Testing environment 1/50 =====
Goals  (1): ['s8']
Traps  (1): ['s6']
Normal (7): 7 cells
  ep 1: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 2: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 3: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 4: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 5: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 6: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 7: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 8: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's7', 's8']
  ep 9: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 10: GOAL in 3 steps
here is the full path in this episode : ['s1', 's4', 's5', 's8']
  ep 11: GOAL

In [ ]:
# in training(), replace free_states with weighted sampling # we have an issue with the trap state that should be corrected and then generalized
free_states    = [s for s in env.states if s.symbol not in ("X","O")]
trap_neighbours = [
    s for s in free_states
    if any(n and n.symbol == "X" for n in s.neighbours.values())
]
print("=" * 40)

for s in trap_neighbours:
    env.current = s

    for a in Actions:

        if s.neighbours[a] == s5:

            exp = env.step(a)

            print(f"Start : {s.name}")
            print(f"Action: {a}")
            print(f"Next  : {exp['next_state'].name}")
            print(f"Reward: {exp['reward']}")
            print(f"Done  : {exp['done']}")
            print()

Start : s2
Action: up
Next  : s5
Reward: -100.0
Done  : True

Start : s4
Action: right
Next  : s5
Reward: -100.0
Done  : True

Start : s6
Action: left
Next  : s5
Reward: -100.0
Done  : True

Start : s8
Action: down
Next  : s5
Reward: -100.0
Done  : True



## testing in an other and bigger env

In [ ]:
# comment or uncomment the ones you want to use or remove

# Classic
#env1 = Chessboard(board=Global_board, actions=Actions, start=b38)

# Stronger reward
#env1 = Chessboard(board=Global_board, actions=Actions, start=b38, reward_fn=reward_v2)

# Distance shaping
env1 = Chessboard(board=Global_board, actions=Actions, start=b38, reward_fn=reward_v4)

In [ ]:
# test the previous agent in the big env

configs = [{"based_env": env1, "n_goals": 3, "n_traps": 10, "number_of_sets": 15}]

test_set = training_set_generator(configs=configs)


Generated 15 environments from 1 configs


In [ ]:

results = test_agent(agent, test_set, max_steps=500, n_episodes=10)


===== Testing environment 1/15 =====
Goals  (3): ['b27', 'b42', 'b55']
Traps  (10): ['b11', 'b13', 'b18', 'b19', 'b29', 'b49', 'b53', 'b54', 'b58', 'b61']
Normal (41): 41 cells
  ep 1: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 2: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 3: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 4: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 5: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 6: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 7: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 8: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b42']
  ep 9: GOAL in 32 steps
here is the full path in this episode : ['b38', 'b35', 'b34', 'b16', 'b15', 'b14', 'b14', 'b17', 'b21', 'b22', 'b21', 'b22', 'b21', 'b22', 'b21', 'b22', 'b21', 'b17'

In [ ]:
generator = EnvironmentGenerator(seed=42)

training_envs_1 = generator.generate(
    configs=[
        # easy: single small box, no traps
        {
            "specs":    [{"geometry": "box", "shape": (3, 3)}],
            "n_pieces": 1,
            "n_goals":  1,
            "n_traps":  0,
            "n_envs":   30
        },
        # medium: single box, some traps
        {
            "specs":    [{"geometry": "box", "shape": (3, 3)}],
            "n_pieces": 1,
            "n_goals":  1,
            "n_traps":  2,
            "n_envs":   40
        },
        # hard: two boxes composed, many traps
        {
            "specs":    [{"geometry": "box", "shape": (3, 3)},
                         {"geometry": "box", "shape": (3, 3)}],
            "n_pieces": 2,
            "n_goals":  2,
            "n_traps":  5,
            "n_envs":   50
        },
        # very hard: large box + line
        {
            "specs":    [{"geometry": "box",   "shape": (9, 9)},
                         {"geometry": "h_line","length": 4}],
            "n_pieces": 2,
            "n_goals":  3,
            "n_traps":  12,
            "n_envs":   30
        },
    ],
    actions = Actions,
    shuffle = True
)



Generated 150 environments from 4 configs


In [ ]:
n_features1 = len(state_to_vector(env1.states[0]))
action_dim  = len(Actions)

net1 = ActorCritic(n_features1 , action_dim , hidden_dim=64 , n_layers= 4 )

optimizer = torch.optim.Adam(net1.parameters(), lr=3e-4)
ppo1 = PPO(
    optimizer= optimizer , 
    clip_epsilon= 0.2 , 
    cv = 0.7 , ce = 0.04 , 
    gamma= 0.99 , 
    lam = 0.95 ,
    n_epochs= 4 
    ) # epsilon = 0.2 , cv = 0.5
                                                                                                       # 4
Pi1 = Policy_PPO(net1,env.actions)

agent_carter = Agent_PPO(ppo1,Pi1)

training_envs_1 = training_set_generator([
    {"based_env": env1, "n_goals": 1, "n_traps": 3,  "number_of_sets": 50}, #30
    {"based_env": env1, "n_goals": 2, "n_traps": 8,  "number_of_sets": 50}, #40
    {"based_env": env1, "n_goals": 3, "n_traps": 12, "number_of_sets": 50}, #30
    {"based_env": env1, "n_goals": 3, "n_traps": 15, "number_of_sets": 50,"type":"blocked"}, #20
])

random.shuffle(training_envs_1)


    
# train
for n, env_1 in enumerate(training_envs_1):
    
    print(f"Training env {n+1}/{len(training_envs_1)}")
    training(
        env            = env_1,
        agent          = agent_carter,
        episodes       = 600,
        steps          = 500,
        rollout_length = 128,
        random_start   = True
    )


# training_multi(

#     env_list=training_envs_1,

#     agent=agent_carter,

#     total_episodes=20000,

#     steps=150,

#     rollout_length=512,

#     random_start=True
# )

print("training over the data set completed")

Generated 200 environments from 4 configs
Training env 1/200
Training env 2/200
Training env 3/200
Training env 4/200
Training env 5/200
Training env 6/200
Training env 7/200
Training env 8/200
Training env 9/200
Training env 10/200
Training env 11/200
Training env 12/200
Training env 13/200
Training env 14/200
Training env 15/200
Training env 16/200
Training env 17/200
Training env 18/200
Training env 19/200
Training env 20/200
Training env 21/200
Training env 22/200
Training env 23/200
Training env 24/200
Training env 25/200
Training env 26/200
Training env 27/200
Training env 28/200
Training env 29/200
Training env 30/200
Training env 31/200
Training env 32/200
Training env 33/200
Training env 34/200
Training env 35/200
Training env 36/200
Training env 37/200
Training env 38/200
Training env 39/200
Training env 40/200
Training env 41/200
Training env 42/200
Training env 43/200
Training env 44/200
Training env 45/200
Training env 46/200
Training env 47/200
Training env 48/200
Trainin

/tmp/ipykernel_43626/4072288854.py:51: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  if torch.isfinite(A.std()) and A.std() > 1e-8:


Training env 55/200
Training env 56/200
Training env 57/200
Training env 58/200
Training env 59/200
Training env 60/200
Training env 61/200
Training env 62/200
Training env 63/200
Training env 64/200
Training env 65/200
Training env 66/200
Training env 67/200
Training env 68/200
Training env 69/200
Training env 70/200
Training env 71/200
Training env 72/200
Training env 73/200
Training env 74/200
Training env 75/200
Training env 76/200
Training env 77/200
Training env 78/200
Training env 79/200
Training env 80/200
Training env 81/200
Training env 82/200
Training env 83/200
Training env 84/200
Training env 85/200
Training env 86/200
Training env 87/200
Training env 88/200
Training env 89/200
Training env 90/200
Training env 91/200
Training env 92/200
Training env 93/200
Training env 94/200
Training env 95/200
Training env 96/200
Training env 97/200
Training env 98/200
Training env 99/200
Training env 100/200
Training env 101/200
Training env 102/200
Training env 103/200
Training env 104

In [ ]:
debug_learning_balance(agent_carter.policy)


LEARNING BALANCE
Actor gradient : 0.022150
Critic gradient: 0.023272
Ratio A/C      : 0.952
Status : Balanced


{'actor': 0.022149658119398216,
 'critic': 0.023272447866480604,
 'ratio': 0.9517541398710807}

In [ ]:
debug_ppo(agent = agent_carter ,env = env1)


PPO COMPLETE DEBUG

[1] ENVIRONMENT
----------------------------------------------------------------------
b11 -- up     --> b14 reward=  -1.0 done=True symbol=X
b11 -- right  --> b12 reward= -0.01 done=False symbol=None
b11 -- down   --> b11 reward= -0.01 done=False symbol=None
b11 -- left   --> b11 reward= -0.01 done=False symbol=None
b12 -- up     --> b15 reward=0.09000000000000001 done=False symbol=None
b12 -- right  --> b13 reward= -0.11 done=False symbol=None
b12 -- down   --> b12 reward= -0.01 done=False symbol=None
b12 -- left   --> b11 reward= -0.01 done=False symbol=None
b13 -- up     --> b16 reward=0.09000000000000001 done=False symbol=None
b13 -- right  --> b31 reward=0.09000000000000001 done=False symbol=None
b13 -- down   --> b13 reward= -0.01 done=False symbol=None
b13 -- left   --> b12 reward=0.09000000000000001 done=False symbol=None
b14 -- up     --> b17 reward=0.09000000000000001 done=False symbol=None
b14 -- right  --> b15 reward= -0.01 done=False symbol=None
b14 -

/tmp/ipykernel_43626/275698226.py:132: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.data.std().item():.6f}")
/tmp/ipykernel_43626/275698226.py:154: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.grad.std().item():.6f}")


In [ ]:
configs = [

    # ========================================
    # RANDOM ENVIRONMENTS
    # ========================================

    #  {
    #      "based_env": env1,
    #      "n_goals": 3,
    #      "n_traps": 12,
    #      "number_of_sets": 30,
    #      "type": "random"
    #  },


    # ========================================
    # BLOCKED ENVIRONMENTS
    # ========================================

    {
        "based_env": env1,
        "n_goals": 3,
        "n_traps": 15,
        "number_of_sets": 500,
        "type": "blocked"
    }

]


test_set_1 = training_set_generator(configs)



Generated 500 environments from 1 configs


In [ ]:

#results = test_agent(agent, test_set, max_steps=50, n_episodes=10)

results = test_agent(agent_carter, test_set_1, max_steps=50, n_episodes=10)


===== Testing environment 1/500 =====
Goals  (3): ['b39', 'b47', 'b64']
Traps  (15): ['b17', 'b21', 'b22', 'b26', 'b27', 'b29', 'b33', 'b34', 'b37', 'b41', 'b42', 'b45', 'b58', 'b61', 'b69']
Normal (36): 36 cells
  ep 1: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b39']
  ep 2: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b39']
  ep 3: GOAL in 7 steps
here is the full path in this episode : ['b38', 'b35', 'b32', 'b35', 'b32', 'b35', 'b38', 'b39']
  ep 4: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b39']
  ep 5: GOAL in 1 steps
here is the full path in this episode : ['b38', 'b39']
  ep 6: TIMEOUT
here is the full path in this episode : ['b38', 'b35', 'b32', 'b31', 'b13', 'b16', 'b15', 'b18', 'b19', 'b18', 'b19', 'b18', 'b19', 'b18', 'b15', 'b18', 'b15', 'b18', 'b15', 'b18', 'b15', 'b14', 'b15', 'b12', 'b15', 'b12', 'b15', 'b18', 'b19', 'b18', 'b19', 'b18', 'b19', 'b16', 'b19', 'b18', 'b19', 'b18', 'b19', 'b18', 'b15', 'b12'

### Potential issue

The only thing to be careful about is that everyone must reference the same network object.

#### Good:
```python
net = ActorCritic(...)

policy = Policy_PPO(net)
optimizer = torch.optim.Adam(net.parameters())

ppo = PPO(network=net, optimizer=optimizer)
```

All three share the same net.

#### Bad:
```python
policy = Policy_PPO(ActorCritic(...))
ppo = PPO(network=ActorCritic(...), ...)
``` 
Now there are two different networks:

policy uses one,
PPO updates another.

The policy will never see the learned weights.

### Recommendation

Pass the same network instance everywhere:
```python
net = ActorCritic(...)

policy = Policy_PPO(net)
optimizer = torch.optim.Adam(net.parameters())
strategy = PPO(net, optimizer)
agent = Agent(strategy, policy)
```
This architecture is clean, modular, and very close to how RL libraries (e.g. Stable-Baselines3, CleanRL) organize actor-critic agents.

In [ ]:
# import the sb3 here and see how to integrate them with my code (see if i can package it under my designed classes)
from stable_baselines3 import ppo

dir(ppo)


['CnnPolicy',
 'MlpPolicy',
 'MultiInputPolicy',
 'PPO',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'policies',
 'ppo']

In [ ]:
# import the gymnasium stuff here
import gym
help(gym.ActionWrapper)

Help on class ActionWrapper in module gym.core:

class ActionWrapper(Wrapper)
 |  ActionWrapper(env: gym.core.Env)
 |
 |  Superclass of wrappers that can modify the action before :meth:`env.step`.
 |
 |  If you would like to apply a function to the action before passing it to the base environment,
 |  you can simply inherit from :class:`ActionWrapper` and overwrite the method :meth:`action` to implement
 |  that transformation. The transformation defined in that method must take values in the base environment’s
 |  action space. However, its domain might differ from the original action space.
 |  In that case, you need to specify the new action space of the wrapper by setting :attr:`self.action_space` in
 |  the :meth:`__init__` method of your wrapper.
 |
 |  Let’s say you have an environment with action space of type :class:`gym.spaces.Box`, but you would only like
 |  to use a finite subset of actions. Then, you might want to implement the following wrapper::
 |
 |      class Discret

In [ ]:
env = gym.make("InvertedPendulum-v2")
dir(gym)

['ActionWrapper',
 'Env',
 'ObservationWrapper',
 'RewardWrapper',
 'Space',
 'Wrapper',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'core',
 'envs',
 'error',
 'logger',
 'make',
 'notice',
 'notices',
 'os',
 'register',
 'spaces',
 'spec',
 'sys',
 'utils',
 'vector',
 'version',
 'wrappers']

In [ ]:
pip install "gymnasium[mujoco]"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
env = gym.make("InvertedPendulum-v2")


/home/mohamed_admin/Desktop/Code-Scraps/venv-fresh/lib/python3.12/site-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment InvertedPendulum-v2 is out of date. You should consider upgrading to version `v4`.
  logger.warn(


DependencyNotInstalled: No module named 'mujoco_py'. (HINT: you need to install mujoco_py, and also perform the setup instructions here: https://github.com/openai/mujoco-py/.)